In [1]:
#!/usr/bin/env python3
import rospy
from actionlib import SimpleActionClient, TerminalState
from assignment_2_2024.msg import PlanningAction, PlanningGoal, PlanningFeedback
from nav_msgs.msg import Odometry
from geometry_msgs.msg import Pose, PoseStamped, Point,Twist
from std_msgs.msg import Float32
from sensor_msgs.msg import LaserScan
import time
import ipywidgets as widgets
from ipywidgets import Button, Layout, HBox, VBox
from IPython.display import display,clear_output,HTML
import matplotlib.pyplot as plt

#import for visualization
import tf
from tf.transformations import quaternion_matrix
import numpy as np
from matplotlib.animation import FuncAnimation

stop_toggle = True
vel :Twist = Twist()
xdata, ydata = [],[]


target_given_list =[]
target_reached_list =[]
target_cancelled_list= []

# variabes to deal with the goal definition
stat: str = ""
pose: Pose = Pose()
goal: PlanningGoal = PlanningGoal()
goal_past : PlanningGoal = PlanningGoal()
    
# lables to debug
lab_pos = widgets.Label(value = "(x: 0.0, y:0.0)")
lab_short_dis = widgets.Label(value =" : 0.0")
arrived_widget = widgets.Output() 
lab_feed = widgets.Label(value = "")

# Generation of buttons
start_b = widgets.Button(description=' Start', layout=Layout(width='50%', height='80px'), disabled =False)
cancel_b = widgets.Button(description='Cancel target', layout=Layout(width='50%', height='80px'),disabled =True)

# display widgets for targets
given_list_output = widgets.Output()
reached_list_output = widgets.Output()
cancelled_list_output = widgets.Output()
given_list_html = widgets.HTML()

# display widgets for the boxes in the table
given_list_box = widgets.VBox([])
reached_list_box =widgets.VBox([])
cancelled_list_box =widgets.VBox([])

# data for obstacle, x and y positions
text_ob = widgets.Text(value = "0.0",layout = widgets.Layout(width = '80px'))
text_x = widgets.Text(value = "0.0", layout = widgets.Layout(width = '80px'))
text_y = widgets.Text(value = "0.0",layout = widgets.Layout(width = '80px'))

# callbacks

In [2]:
def pos_vel_callcback(msg):
    global vel , xdata, ydata
    vel = msg.twist.twist
    lab_pos.value = f'(x: {msg.pose.pose.position.x :.2f}, y:{msg.pose.pose.position.y :.2f})'
    text_x.value = f'{msg.pose.pose.position.x:.2f}'
    text_y.value = f'{msg.pose.pose.position.y:.2f}'
    xdata.append( msg.pose.pose.position.x)
    ydata.append( msg.pose.pose.position.y)
    
def feedback_callback(feedback_: PlanningFeedback) -> None:
    global stat
    stat = feedback_.feedback.stat
    process_feedback()

def laser_callback(msg):
    # retrieve the data from laserscan and process it to obtain the closest obstacle
    global min_distance
    min_distance = float('inf')
    for range in msg.ranges:
        if range < min_distance:
            min_distance = range
    closest_obstacle_msg = Float32()
    closest_obstacle_msg.data = min_distance
    publish_distance.publish(closest_obstacle_msg)
    
    # update the display
    lab_short_dis.value = f' {min_distance:.2f}'
    text_ob.value = f'{min_distance:.3f}'

# buttons behaviours

In [3]:
def on_start_button_clicked(b):
    global x_input, y_input
    try:
        # checking on conditions 
        if (-9 <= x_input.value <= 9) and (-9 <= y_input.value <= 9) and not (
                (abs(x_input.value) == 9 and abs(y_input.value) == 9) or
                (abs(x_input.value) == 8 and abs(y_input.value) == 9) or
                (abs(x_input.value) == 9 and abs(y_input.value) == 8) or
                (abs(x_input.value) == 8 and abs(y_input.value) == 8)
            ):
            
            # valid target : no corners and not triagle (+-8, +-9)  and (+-9, +-8) cominations
            warning_msg.value = ""
            start_b.disabled = True
            cancel_b.disabled = False
            x_input.disabled = True
            y_input.disabled = True
            
            # send the goal
            do_goal(x_input,y_input)
            
        elif (abs(x_input.value) == 9 and abs(y_input.value) == 9):
            warning_msg.value = "<span style='color:red;'>Not admissable reagion:<br> extreme corners</span>"
            start_b.disabled = False
        elif  ( (abs(x_input.value) == 8 and abs(y_input.value) == 9) or
                (abs(x_input.value) == 9 and abs(y_input.value) == 8) or
                (abs(x_input.value) == 8 and abs(y_input.value) == 8)):
            warning_msg.value = f"<span style='color:red;'>Not admissable reagion:<br> {x_input.value, y_input.value} not feasible  combination </span>"
            start_b.disabled = False
    except ValueError:
        warning_msg.value = "<span style='color:red;'>Not amissable reagion</span>"
        start_b.disabled = False
    
    
    
    


    
def on_cancel_button_clicked(b):
    global cancel_b,start_b,stop_toggle
    cancel_b.disabled = True
    start_b.disabled =False
    x_input.disabled = False
    y_input.disabled = False
    x_input.value = 0.0
    y_input.value = 0.0
    del_goal()


# Functions for the robot 

In [4]:
def process_feedback():
    global stat, target_reached_list, target_given_list,given_list_html
           
    if(stat == "Target reached!"):
        
        # add the last element of target_given_list in the list of the targets reached 
        target_reached_list.append(target_given_list[-1])  
        
        # toggle the buttons and reset the interface 
        cancel_b.disabled = True
        start_b.disabled =False
        x_input.disabled = False
        y_input.disabled = False
        x_input.value = 0.0
        y_input.value = 0.0
        
        # add the target to reached display and add a blanck row in cancelled targets,
        # then color from main list display the green target as reched
        new_target_rea = widgets.HTML(f'<span style="color: black;">{target_reached_list[-1]}</span>')
        reached_list_box.children += (new_target_rea,)
        new_target_can = widgets.HTML(f'<br>')
        cancelled_list_box.children += (new_target_can,)
        given_list_box.children[-1].value = f'<span style="color: green;">{target_given_list[-1]}</span>'
        
        # clear the goal
        del_goal()
            
    if(stat == "Target cancelled!" ):
        
        # clear the goal
        del_goal()
        
        # add the target to cancelled display and add a blanck row in reached targets,
        # then color from main list display the red target as reched
        target_cancelled_list.append(target_given_list[-1])
        new_target_rea = widgets.HTML(f'<br>')
        reached_list_box.children += (new_target_rea,)
        new_target_can = widgets.HTML(f'<span style="color: black;">{target_cancelled_list[-1]}</span>')
        cancelled_list_box.children += (new_target_can,)
        given_list_box.children[-1].value = f'<span style="color: red;">{target_given_list[-1]}</span>'

    
def do_goal(x_input,y_input):
    global pose, goal, client, target_given_list, given_list_html, given_list_box
    
    # clear current possible goals
    del_goal()
    
    # build the target 
    pose.position.x = x_input.value
    pose.position.y = y_input.value
    target = {'aim_x': 0.0, 'aim_y': 0.0}
    target['aim_x'] = pose.position.x
    target['aim_y'] = pose.position.y
    target_given_list.append(target)
    
    # add to display
    new_target = widgets.HTML(f'<span style="color: black;">{target}</span>')
    given_list_box.children += (new_target,)
    with given_list_output:
        print(target_given_list) 
        
    # create the goal
    goal.target_pose.pose = pose
    
    # send the goal
    client.send_goal(goal)


def del_goal():
    global client,stop_toggle
    client.cancel_all_goals()


# Plot number reached and non reached targets and plot of position of the robot

In [5]:
def update_plots(frame):
    global target_reached_list, target_cancelled_list, target_given_list, xdata,ydata
    if ((len(target_reached_list) != 0 or len(target_cancelled_list) != 0)):
        ax1.clear()
        sizes = [len(target_reached_list), len(target_cancelled_list)]
        labels = ['Reached', 'Cancelled']
        explode = (0.1, 0) if sum(sizes) > 0 else None
        ax1.pie(sizes, labels=labels, explode=explode, autopct='%1.1f%%' )
        ax1.set_title(f'Reached vs Cancelled \n  sent: {len(target_given_list)}, reached: {len(target_reached_list)}, cancelled: {len(target_cancelled_list)}', fontsize = 10)
    ln.set_data(xdata, ydata)
    ax2.set_xlim(10, -10)
    ax2.set_ylim(10, -10)
    return ln,

%matplotlib widget

plt.ioff()
fig, (ax2, ax1) = plt.subplots(1, 2, figsize=(9, 5))  

# Pie chart (ax1)
fig.patch.set_visible(False)
ax1.set_title(
    f'Reached vs Cancelled\nsent: {len(target_given_list)}, '
    f'reached: {len(target_reached_list)}, cancelled: {len(target_cancelled_list)}',
    fontsize=10
)
sizes = [len(target_reached_list), len(target_cancelled_list)]
# Trajectory plot (ax2)

ax2.grid()
ln, = ax2.plot([], [], 'r-')
ax2.set_xlim(10, -10)
ax2.set_ylim(10, -10)
ax2.set_title("Live trajectory")
ax2.set_xlabel("x [m]")
ax2.set_ylabel("y [m]")
ax2.yaxis.set_label_position("right")
plt.ion()




# Interface

In [6]:
rospy.init_node("target_client")
client: SimpleActionClient = SimpleActionClient("reaching_goal", PlanningAction)
fdk= rospy.Subscriber("/reaching_goal/feedback", PlanningFeedback, feedback_callback , queue_size=10)
rospy.Subscriber("/odom", Odometry, pos_vel_callcback)
pub_vel = rospy.Publisher("/cmd_vel", Twist,queue_size = 10 )
rospy.Subscriber('/scan', LaserScan, laser_callback)
publish_distance = rospy.Publisher('closest_obstacle', Float32, queue_size=10)
del_goal()

# Input boxes for target
message_label = widgets.HTML(
    value='<h2 style="margin:0px; padding-left:0px;">Set target (x,y):</h2>',
    layout=widgets.Layout(display='flex', align_items='center')
)
x_input = widgets.FloatText(value=0.0,  layout=widgets.Layout(width='80px'))
y_input = widgets.FloatText(value=0.0,  layout=widgets.Layout(width='80px'))

warning_msg = widgets.HTML(
    value="",
    layout=widgets.Layout(margin='10px 0px 0px 10px', width='300px') 
)


box_input = HBox(
    [message_label, x_input, y_input],
    layout=widgets.Layout(
        justify_content='flex-start', 
        align_items='center',      
        width='100%'               
    )
)




# Generation of buttons
start_b = widgets.Button(description=' Start', layout=Layout(width='50%', height='80px'), disabled =False, button_style='success')
cancel_b = widgets.Button(description='Cancel target', layout=Layout(width='50%', height='80px'),disabled =True, button_style='danger')


# Output widget
output_start = widgets.Output()
output_cancel = widgets.Output()


# binding buttons and functions
start_b.on_click(on_start_button_clicked)
cancel_b.on_click(on_cancel_button_clicked)

# texts for the position 
text_x.disabled = True
text_y.disabled = True
label_pos = widgets.HTML(
    value='<h2 style="margin:0; padding-right:10px;">Position (x,y):</h2>',
    layout=widgets.Layout(display='flex', align_items='center')
)
box_pos = HBox(
    [label_pos, text_x, text_y],
    layout=widgets.Layout(
        justify_content='flex-start', 
        align_items='center',      
        width='100%'               
    )
)

#text for closest obstacle
text_ob.disabled = True
label1_ob = widgets.HTML(
    value='<h2 style="margin:0px; padding-right:10px;">Closest obstacle:</h2>',
    layout=widgets.Layout(display='flex', align_items='center')
    
)

label2_ob = widgets.HTML(
    value='<h2 style="margin:0px; padding-left:10px;">metres</h2>',
    layout=widgets.Layout(display='flex', align_items='center')
)

box_ob = HBox(
    [label1_ob, text_ob, label2_ob],
    layout=widgets.Layout(justify_content='flex-start', align_items='center', width='100%')
)





# Widget visualization


display(widgets.HBox([box_input, warning_msg]))
display(HBox( [start_b,  cancel_b]))
out1 = widgets.Output()
out2 = widgets.Output()
display(HBox([box_pos,box_ob]))
plt.show()
ani = FuncAnimation(fig, update_plots, interval=100)

display(lab_feed)

box_layout = Layout(border='1px solid black', width='33%', align_items='center', padding='10px')
display(HBox([
    VBox([widgets.HTML(f'<b> Target cancelled </b>'), cancelled_list_box], layout=box_layout),
    VBox([widgets.HTML(f'<b> Target given </b>'), given_list_box], layout=box_layout),
    VBox([widgets.HTML(f'<b> Target reached </b>'), reached_list_box], layout=box_layout)
], layout=Layout(width='100%')))


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Label(value='')